# Exercises XP: Time Series with LSTM

This is a guided notebook for the exercise on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear for important concepts.

## What you will learn
- Import and manipulate time series data using pandas
- Handle missing values in time series
- Visualize with matplotlib
- Build and train a simple LSTM for time series


## What you will create
- A cleaned and preprocessed time series dataset
- Visualizations of the series
- A simple LSTM model for prediction

# Part 1: Data Import and Initial Exploration

**Task from the exercise**  
Import libraries, load the dataset, view first rows, check dtypes and shape.

**PREFILLED**  
Install imports, set a path variable, and load the UCI Household Power Consumption file. The file is usually semicolon separated with `?` as missing.

In [ ]:
# PREFILLED: just execute
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 100)

# Dataset is in the same folder as the notebook
DATA_PATH = Path('./household_power_consumption.txt')

# Robust loader for semicolon separated file with ? as NA
parse_cols = {
    'Global_active_power': 'float64',
    'Global_reactive_power': 'float64',
    'Voltage': 'float64',
    'Global_intensity': 'float64',
    'Sub_metering_1': 'float64',
    'Sub_metering_2': 'float64',
    'Sub_metering_3': 'float64',
}
usecols = ['Date', 'Time'] + list(parse_cols.keys())

df = pd.read_csv(
    DATA_PATH,
    sep=';',
    na_values=['?'],
    usecols=usecols,
    dtype=parse_cols,
    parse_dates={'Datetime': ['Date', 'Time']},
    infer_datetime_format=True,
    dayfirst=True,
)

df = df.set_index('Datetime').sort_index()
print('Shape:', df.shape)
display(df.head())
display(df.dtypes)

# Part 2: Handling Missing Values

**Task from the exercise**  
Identify columns with missing values. Fill missing values using the mean of each column. Verify there are no missing values left.

**To-Do (code):** Compute a missing value summary, then fill numeric columns with their mean. Recheck for remaining NaNs.

In [ ]:
# To-Do: show missing summary, impute with column means, verify

# ── Step 1: identify columns with missing values ──────────────────────────────
missing_before = df.isna().sum().sort_values(ascending=False)
print('Missing values per column (before fill):')
display(missing_before[missing_before > 0])

# ── Step 2: fill numeric columns with their column mean ───────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# ── Step 3: verify no NaNs remain ─────────────────────────────────────────────
missing_after = df.isna().sum().sum()
print(f'\nTotal missing values after fill: {int(missing_after)}')
print('All missing values handled.' if missing_after == 0 else 'WARNING: NaNs still present!')

# Part 3: Data Visualization

**Task from the exercise**  
Resample the `Global_active_power` column over a day and plot the sum and mean. Plot the mean and standard deviation of `Global_intensity` resampled over a day.

**PREFILLED**  
Helper plotting functions using matplotlib.

In [ ]:
# PREFILLED: just execute
def plot_daily_sum_and_mean(series, title_prefix='Global_active_power'):
    daily = series.resample('1D')
    s = daily.sum(min_count=1)
    m = daily.mean()
    plt.figure(figsize=(8, 3))
    plt.plot(s.index, s.values)
    plt.title(f'{title_prefix} daily sum')
    plt.xlabel('date'); plt.ylabel('sum'); plt.tight_layout(); plt.show()
    plt.figure(figsize=(8, 3))
    plt.plot(m.index, m.values)
    plt.title(f'{title_prefix} daily mean')
    plt.xlabel('date'); plt.ylabel('mean'); plt.tight_layout(); plt.show()

def plot_daily_mean_std(series, title_prefix='Global_intensity'):
    daily = series.resample('1D')
    m  = daily.mean()
    sd = daily.std()
    plt.figure(figsize=(8, 3))
    plt.plot(m.index,  m.values,  label='mean')
    plt.plot(sd.index, sd.values, label='std')
    plt.title(f'{title_prefix} daily mean and std')
    plt.xlabel('date'); plt.ylabel('value')
    plt.legend(); plt.tight_layout(); plt.show()

**To-Do (code):** Call the helpers on the required columns. If a plot seems too dense, slice a recent time window.

In [ ]:
# To-Do: visualize daily aggregations

# Plot daily sum and mean of Global_active_power
plot_daily_sum_and_mean(df['Global_active_power'], title_prefix='Global_active_power')

# Plot daily mean and std of Global_intensity
plot_daily_mean_std(df['Global_intensity'], title_prefix='Global_intensity')

# Part 4: Data Preprocessing for LSTM

**Task from the exercise**  
Normalize the dataset, split into train and test, and reshape for LSTM input.

**Learning point**  
LSTMs consume sequences with shape [batch, time, features]. You must build sliding windows. Use a lookback window and predict the next step.

![image.png](https://github.com/user-attachments/assets/be3e815f-2aff-4e8d-ac48-eca27b1c729c)

**PREFILLED**  
Windowing helpers and a train test split by time.

In [ ]:
# PREFILLED: just execute
from sklearn.preprocessing import MinMaxScaler

def make_windows(arr_2d, window=48, horizon=1):
    X, y = [], []
    for i in range(len(arr_2d) - window - horizon + 1):
        X.append(arr_2d[i:i + window])
        y.append(arr_2d[i + window:i + window + horizon, 0])
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    return X, y

def time_train_test_split(df, split_ratio=0.8):
    n = len(df)
    k = int(n * split_ratio)
    return df.iloc[:k].copy(), df.iloc[k:].copy()

feat_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity',
             'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df_feats = df[feat_cols].copy()

df_tr, df_te = time_train_test_split(df_feats, split_ratio=0.8)

scaler    = MinMaxScaler()
tr_scaled = scaler.fit_transform(df_tr.values)
te_scaled = scaler.transform(df_te.values)

WINDOW  = 48
HORIZON = 1
X_tr, y_tr = make_windows(tr_scaled, window=WINDOW, horizon=HORIZON)
X_te, y_te = make_windows(te_scaled, window=WINDOW, horizon=HORIZON)

print('Train windows:', X_tr.shape, '  Train targets:', y_tr.shape)
print('Test  windows:', X_te.shape, '  Test  targets:', y_te.shape)

> **To-Do (write here):** State which column you predict as the target and why. If you changed `feat_cols`, note your choice.

The target column is **`Global_active_power`** (column index 0 of `feat_cols`), which is the first column extracted by `make_windows` via `arr_2d[..., 0]`.

**Why `Global_active_power`?**  
`Global_active_power` represents the total active power consumed by the household in kilowatts — it is the most direct and informative measure of overall energy demand. Predicting it allows us to forecast future electricity consumption, which is the core objective in household energy analytics. The other columns (reactive power, voltage, intensity, sub-meters) are kept as input features because they correlate with and help explain variations in active power.

# Part 5: Building an LSTM Model

**Task from the exercise**  
Import the libraries, define the LSTM architecture, and compile with a suitable loss and optimizer.

**To-Do (code):** Build a small LSTM with one or two LSTM layers and a Dense output of size `HORIZON`. Use `mse` loss and `adam` optimizer. Print `model.summary()`.

In [ ]:
# To-Do: define and compile the LSTM
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)

model = models.Sequential([
    layers.Input(shape=(WINDOW, X_tr.shape[-1])),

    # First LSTM layer — return sequences so the second layer gets the full sequence
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),

    # Second LSTM layer — only the last hidden state is passed forward
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),

    # Dense output: one value per horizon step
    layers.Dense(HORIZON),
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

# Part 6: Training and Evaluating the LSTM Model

**Task from the exercise**  
Train the model, evaluate on the test set, and plot training and validation loss.

**PREFILLED**  
Callbacks and plotting helper. You supply the model and data in the To-Do cell.

In [ ]:
# PREFILLED: just execute
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def fit_and_plot(model, X_tr, y_tr, X_te, y_te, epochs=10, batch=256):
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    rl = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
    h  = model.fit(
        X_tr, y_tr,
        validation_data=(X_te, y_te),
        epochs=epochs,
        batch_size=batch,
        callbacks=[es, rl],
        verbose=2
    )
    plt.figure(figsize=(6, 4))
    plt.plot(h.history['loss'],     label='loss')
    plt.plot(h.history['val_loss'], label='val_loss')
    plt.title('Training and validation loss')
    plt.xlabel('epoch'); plt.ylabel('loss')
    plt.legend(); plt.tight_layout(); plt.show()
    return h

**To-Do (code):** Train and evaluate. Report final test MAE and MSE.

In [ ]:
# To-Do: train and evaluate

# Train the model with early stopping and learning rate reduction
hist = fit_and_plot(model, X_tr, y_tr, X_te, y_te, epochs=15, batch=256)

# Evaluate on the test set
eval_mse, eval_mae = model.evaluate(X_te, y_te, verbose=0)
print({'test_mse': float(eval_mse), 'test_mae': float(eval_mae)})